In [1]:
%%capture
%uv pip install unsloth
# Also get the latest nightly Unsloth!
%uv pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git@nightly git+https://github.com/unslothai/unsloth-zoo.git

In [15]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = torch.bfloat16  # Changed from None
load_in_4bit = False  # Changed from True - B200 has enough memory!

model, tokenizer = FastLanguageModel.from_pretrained(
  model_name = "unsloth/Llama-3.2-3B-Instruct",
  max_seq_length = max_seq_length,
  dtype = dtype,
  load_in_4bit = load_in_4bit,
)


==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.56.0.
   \\   /|    NVIDIA B200. Num GPUs = 1. Max memory: 178.351 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 10.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [16]:
model = FastLanguageModel.get_peft_model(
  model,
  r = 64,  # Increased from 16 - B200 can handle it
  target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",
                    "embed_tokens", "lm_head"],  # Added embed & lm_head
  lora_alpha = 128,  # 2x the rank
  lora_dropout = 0,
  bias = "none",
  use_gradient_checkpointing = False,  # Disabled for speed
  random_state = 3407,
  use_rslora = True,  # Rank-stabilized LoRA - better convergence
  loftq_config = None,
)

/usr/local/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1222: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


In [4]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [5]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched = True,)

Unsloth: Standardizing formats (num_proc=115):   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [6]:
dataset[5]["conversations"]

[{'content': 'How do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?',
  'role': 'user'},
 {'content': 'Astronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.',
  'role': 'assistant'}]

In [7]:
dataset[5]["text"]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHow do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAstronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.<|

In [17]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
  model = model,
  tokenizer = tokenizer,
  train_dataset = dataset,
  dataset_text_field = "text",
  max_seq_length = max_seq_length,
  data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
  dataset_num_proc = 8,  # Increased from 2
  packing = True,  # ENABLED - huge speedup!
  args = TrainingArguments(
      per_device_train_batch_size = 32,  # Increased from 2
      gradient_accumulation_steps = 4,   # Effective batch = 128

      # Full epoch instead of max_steps
      num_train_epochs = 1,

      # Learning rate & schedule
      learning_rate = 2e-4,
      lr_scheduler_type = "cosine",  # Changed from linear
      warmup_ratio = 0.03,  # Changed from warmup_steps

      # Precision
      bf16 = True,
      fp16 = False,
      tf32 = True,  # Enable TensorFloat-32

      # Optimizer
      optim = "adamw_torch_fused",  # Changed from adamw_8bit
      weight_decay = 0.01,

      # Logging & checkpointing
      logging_steps = 10,  # Less frequent logging
      save_strategy = "steps",
      save_steps = 100,  # Adjust based on total steps
      save_total_limit = 3,

      # Performance
      dataloader_num_workers = 4,  # Enable parallel data loading
      dataloader_pin_memory = True,  # Faster GPU transfers

      # Misc
      seed = 3407,
      output_dir = "outputs",
      report_to = "none",  # Consider "wandb" for monitoring
  ),
)

Unsloth: Tokenizing ["text"] (num_proc=119):   0%|          | 0/100000 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [18]:
trainer_stats = trainer.train()
# trainer_stats = trainer.train(resume_from_checkpoint="outputs/checkpoint-40")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 782
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 4 x 1) = 128
 "-____-"     Trainable parameters = 885,260,288 of 4,098,010,112 (21.60% trained)


Step,Training Loss
10,1.115400
20,0.826500
30,0.799100
40,0.793300
50,0.765300
60,0.777500
70,0.762700
80,0.774600
90,0.762600
100,0.770600


In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
messages = [                    # Change below!
    {"role": "user", "content": "Continue the fibonacci sequence! Your input is 1, 1, 2, 3, 5, 8,"},
]
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids, streamer = text_streamer, max_new_tokens = 128, pad_token_id = tokenizer.eos_token_id)

In [11]:
# model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)

In [1]:
import os
model.push_to_hub_merged("tegelstenen/model", tokenizer, save_method = "merged_16bit", token = os.getenv("WRITE_HF_TOKEN"))

NameError: name 'model' is not defined